In [1]:
# 读取fa文件，统计不同序列出现的次数以及对应的overall_confidence的均值，最小值，最大值，保存到csv文件中
import pandas as pd
import os
from collections import defaultdict
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]

for pdb in pdbs:
    method_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/LigandMPNN-Output-pred_dimer/AF3-2000-0.1T/'

    pdb_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/alphafold3/PepSet_AF3_pass"
    pdb_list = os.listdir(pdb_path)
    # print(pdb_list)
    if f'{pdb}.pdb' in pdb_list:
        #读取pdb，将A链的单字母序列保存到变量sequence中，注意pdb只有ATOM或HETATM行
        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb, f'{pdb_path}/{pdb}.pdb')
        for model in structure:
            for chain in model:
                if chain.id == 'A':
                    pro_sequence = ''
                    for residue in chain:
                        if residue.id[0] == ' ':
                            one_letter_resname = seq1(residue.get_resname())
                            pro_sequence += one_letter_resname

    for seed in ['seed42', 'seed43', 'seed44']:
        fa_file = f'{method_path}/{pdb}/{seed}/seqs/{pdb}.fa'
        sequences = []
        confidences = defaultdict(list)
        with open(fa_file, 'r') as f:
            for i, line in enumerate(f):
                if i >= 2:
                    if line.startswith('>'):
                        parts = line.strip().split(',')
                        overall_confidence = float(parts[4].split('=')[1])
                        ligand_confidence = float(parts[5].split('=')[1])
                        seq_rec = float(parts[6].split('=')[1])
                    else:
                        seq = line.strip()
                        sequences.append(seq)
                        confidences[seq].append([overall_confidence, ligand_confidence, seq_rec])
        sequence_stats = []
        for seq, conf_list in confidences.items():
            count = len(conf_list)
            avg_overall_confidence = round(sum([conf[0] for conf in conf_list]) / count, 3)
            min_overall_confidence = min([conf[0] for conf in conf_list])
            max_overall_confidence = max([conf[0] for conf in conf_list])
            avg_ligand_confidence = round(sum([conf[1] for conf in conf_list]) / count, 3)
            min_ligand_confidence = min([conf[1] for conf in conf_list])
            max_ligand_confidence = max([conf[1] for conf in conf_list])
            avg_seq_rec = round(sum([conf[2] for conf in conf_list]) / count, 3)

            sequence_stats.append((pro_sequence, seq, count, avg_overall_confidence, min_overall_confidence, max_overall_confidence, avg_ligand_confidence, min_ligand_confidence, max_ligand_confidence, avg_seq_rec))



        df = pd.DataFrame(
            sequence_stats,
            columns=['Pro_Sequence', 'Pep_Sequence', 'Count', 'Average_Overall_Confidence', 'Min_Overall_Confidence', 'Max_Overall_Confidence', 'Average_Ligand_Confidence', 'Min_Ligand_Confidence', 'Max_Ligand_Confidence', 'Average_Seq_Rec']
        )
        df = df.sort_values(by='Max_Overall_Confidence', ascending=False)
        df.index = range(1, len(df) + 1)
        # os.makedirs(f'{method_path}/ranked_by_Max_Overall_Confidence', exist_ok=True)
        df.to_csv(f'{method_path}/{pdb}/{seed}/ranked_by_Max_Overall_Confidence.csv', index=True)


In [2]:
# 读取csv文件，将其中sequence列的序列提取出来，保存在一个list中
import os
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
import argparse
import pandas as pd
import json
import os


json_temple_path = "protenix_template.json"

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]


#根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件


for pdb in pdbs:

    ori_pdb = pdb.split('_')[0]

    path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/LigandMPNN-Output-pred_dimer/AF3-2000-0.1T/{pdb}'
    for seed in ['seed42', 'seed43', 'seed44']:
        jobs = []                                     # 循环内清空jobs
        with open(json_temple_path, 'r') as file:     # 不能在循环外打开，否则后续seed43和seed44会包含前面循环的内容，即每一次循环需要重新读取template.json
            tmpl = json.load(file)
        df = pd.read_csv(f'{path}/{seed}/ranked_by_Max_Overall_Confidence.csv')

        ori_msa_pro_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/MSA_pro_all_PepSet_dimer/{ori_pdb}'
        pairing_db = 'uniref100'
        pep_base_dir = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/predict/Protenix/PepSet_AF3_pass-2k-0.1/{pdb}'  # full path
        for _, row in df.iloc[:10].iterrows():
            name = row.iloc[0]
            seq_pro = row.iloc[1]
            seq_pep = row.iloc[2]
            job_name = pdb + "_" + str(name)

            # 编写多肽的msa文件
            msa = f'{pep_base_dir}/ligandmpnn_{seed}/{job_name}/msa'
            msa_pep_path = f'{msa}/msa_pep'
            msa_pro_path = f'{msa}/msa_pro'
            os.makedirs(msa_pep_path, exist_ok=True)
            os.makedirs(msa_pro_path, exist_ok=True)
            with open(f'{msa_pep_path}/pairing.a3m', 'w') as f:
                f.write('>query\n')
                f.write(seq_pep + '\n')
            os.system(f'cp {msa_pep_path}/pairing.a3m {msa_pep_path}/non_pairing.a3m')

            os.system(f'ln -s {ori_msa_pro_path}/pairing.a3m {msa_pro_path}/pairing.a3m')
            os.system(f'ln -s {ori_msa_pro_path}/non_pairing.a3m {msa_pro_path}/non_pairing.a3m')

            job = json.loads(json.dumps(tmpl[0]))
            job['name'] = job_name
            job['sequences'][0]['proteinChain']['sequence'] = seq_pro
            job['sequences'][0]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pro_path
            job['sequences'][0]['proteinChain']['msa']['pairing_db'] = pairing_db

            job['sequences'][1]['proteinChain']['sequence'] = seq_pep
            job['sequences'][1]['proteinChain']['msa']['precomputed_msa_dir'] = msa_pep_path
            job['sequences'][1]['proteinChain']['msa']['pairing_db'] = pairing_db
            jobs.append(job.copy())


        with open(f'{pep_base_dir}/ligandmpnn_{seed}/pred.json', 'w') as f:
            f.write(json.dumps(jobs, indent=4))

In [7]:
import json
import os
import pandas as pd


af3_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/predict/AlphaFold3/PepSet_AF3_pass-2k_0.1"
# ptx_path = "/home/junjiechen/1_work/LZ/protenix/protenix-260318-alphafold_wdr5-rmCK"
mpnn_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/LigandMPNN-Output-pred_dimer/AF3-2000-0.1T"

with open('/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/datasets/PepSet_AF3_pass_processed/PDB.list', 'r') as f:
    pdbs = [line.strip() for line in f.readlines()]

os.makedirs(af3_path, exist_ok=True)
json_temple_path = "/home/junjiechen/1_work/LZ/alphafold/af_template.json"

for pdb in pdbs:
    for seed in ['seed42', 'seed43', 'seed44']:
        df = pd.read_csv(f'{mpnn_path}/{pdb}/{seed}/ranked_by_Max_Overall_Confidence.csv')
        pep_sequences = df["Pep_Sequence"].tolist()
        pro_sequence = df["Pro_Sequence"].tolist()[0]
        msa_pro_path = f'/home/junjiechen/1_work/250401-Dpepalign/Benchmark/protenix/PepSet/MSA_pro_all_PepSet_dimer/{pdb}'

        for i, pep_sequence in enumerate(pep_sequences[:10]):
            os.makedirs(f"{af3_path}/inputs/ligandmpnn_{seed}", exist_ok=True)
            os.makedirs(f"{af3_path}/outputs/ligandmpnn_{seed}", exist_ok=True)
            
            msa_pep_path = f"{af3_path}/pep_msa/{pdb}/{seed}/{i+1}"
            os.system(f'mkdir -p {msa_pep_path}/')
            with open(f'{msa_pep_path}/pairing.a3m', 'w') as p, open(f'{msa_pep_path}/non_pairing.a3m', 'w') as np:
                p.write(f'>query\n{pep_sequence}\n')
                np.write(f'>query\n{pep_sequence}\n')

            #根据template.json的内容以及fasta文件，编写json文件，将fasta中的pro序列替换template.json中的pro_sequence，将fasta中的pep序列替换template.json中的pep_sequence，保存为新的json文件
            with open(json_temple_path, 'r') as file:
                job = json.load(file)

            seq_pep = pep_sequence
            seq_pro = pro_sequence
            
            # 如果msa存在，则替换msa路径和pairing_db，如果不存在，则不添加msa路径和pairing_db
            job['sequences'][0]['protein']['pairedMsaPath'] = msa_pro_path + "/pairing.a3m"
            job['sequences'][0]['protein']['unpairedMsaPath'] = msa_pro_path + "/non_pairing.a3m"
            job['sequences'][1]['protein']['pairedMsaPath'] = msa_pep_path + "/pairing.a3m"
            job['sequences'][1]['protein']['unpairedMsaPath'] = msa_pep_path + "/non_pairing.a3m"

            job['name'] = pdb + "_" + str(i+1)
            job['sequences'][0]['protein']['sequence'] = seq_pro
            job['sequences'][1]['protein']['sequence'] = seq_pep
            job['modelSeeds'] = [42,43,44]

            with open(f'{af3_path}/inputs/ligandmpnn_{seed}/{pdb}_{i+1}.json', 'w') as f:
                f.write(json.dumps(job, indent=4))


In [8]:
import glob
import json
import os
import pandas as pd

# 仅做一致性检查，不改写任何JSON
if 'af3_path' not in globals():
    af3_path = "/home/junjiechen/1_work/250401-Dpepalign/Benchmark/ligandmpnn/check_filter/predict/AlphaFold3/PepSet_AF3_pass-2k_0.1"


def read_query_sequence(a3m_path: str) -> str:
    with open(a3m_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('>'):
                continue
            return line
    return ""


records = []
total = 0
matched = 0
mismatched = 0
errors = 0

for seed in ["seed42", "seed43", "seed44"]:
    json_pattern = f"{af3_path}/inputs/ligandmpnn_{seed}/*.json"
    for json_file in sorted(glob.glob(json_pattern)):
        total += 1
        try:
            with open(json_file, 'r') as f:
                job = json.load(f)

            job_name = job.get('name', os.path.basename(json_file).replace('.json', ''))
            b_chain = job['sequences'][1]['protein']
            b_seq = b_chain['sequence'].strip()
            msa_path = b_chain['pairedMsaPath']

            if not os.path.exists(msa_path):
                errors += 1
                records.append({
                    'status': 'error',
                    'seed': seed,
                    'json_file': json_file,
                    'job_name': job_name,
                    'b_sequence': b_seq,
                    'msa_sequence': '',
                    'msa_path': msa_path,
                    'message': 'pairedMsaPath not found'
                })
                continue

            msa_seq = read_query_sequence(msa_path).strip()

            if b_seq == msa_seq:
                matched += 1
            else:
                mismatched += 1
                records.append({
                    'status': 'mismatch',
                    'seed': seed,
                    'json_file': json_file,
                    'job_name': job_name,
                    'b_sequence': b_seq,
                    'msa_sequence': msa_seq,
                    'msa_path': msa_path,
                    'message': 'B chain sequence != MSA query sequence'
                })

        except Exception as e:
            errors += 1
            records.append({
                'status': 'error',
                'seed': seed,
                'json_file': json_file,
                'job_name': '',
                'b_sequence': '',
                'msa_sequence': '',
                'msa_path': '',
                'message': str(e)
            })

print(f"Checked: {total}")
print(f"Matched: {matched}")
print(f"Mismatched: {mismatched}")
print(f"Errors: {errors}")

report_df = pd.DataFrame(records)
if not report_df.empty:
    report_path = f"{af3_path}/b_chain_msa_check_report.csv"
    report_df.to_csv(report_path, index=False)
    print(f"Report saved to: {report_path}")
    display(report_df.head(20))
else:
    print("No mismatch/error found.")

Checked: 3300
Matched: 3300
Mismatched: 0
Errors: 0
No mismatch/error found.
